# 02 - Clean the raw job data

**Goal:** turn the raw Glassdoor file into one tidy table, `data/clean/jobs_clean.csv`.

Run the cells **from top to bottom** with **Shift + Enter**. After each cell, read the printed output and check it looks sensible.
The raw file in `data/raw/` is never modified.

| Step | What we fix |
|---|---|
| 1 | Load the file |
| 2 | Remove duplicate rows |
| 3 | Replace placeholder values (`--`) with proper blanks |
| 4 | Drop rows with no job description |
| 5 | Group job titles into clear roles and remove non-analyst jobs |
| 6 | Standardise cities (Bengaluru, Delhi NCR...) |
| 7 | Convert salary text into numbers (annual INR and LPA) |
| 8 | Work out experience level (Entry / Mid / Senior) |
| 9 | Flag suspicious salaries |
| 10 | Save the clean file |

## Step 1 - Load the raw file

In [ ]:
import re
import glob
import unicodedata
import numpy as np
import pandas as pd

files = glob.glob("../data/raw/*.csv")
print("CSV files found:", files)

RAW_FILE = files[0]   # if you have more than one CSV in data/raw, type the right file path here instead

try:
    df = pd.read_csv(RAW_FILE)
except UnicodeDecodeError:
    df = pd.read_csv(RAW_FILE, encoding="latin-1")

print("Rows loaded:", len(df))
df.head(3)

## Step 2 - Remove duplicate rows
The same posting can be scraped twice. Counting it twice would inflate every number in the dashboard.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} rows left")

## Step 3 - Replace placeholders with real blanks
This dataset uses `--` and `Unknown / Non-Applicable` when a value is missing. We turn them into proper blanks (NaN) and trim stray spaces.

In [ ]:
df = df.replace({"--": np.nan, "Unknown / Non-Applicable": np.nan})

for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
        df[col] = df[col].str.strip()

df[["company_founded", "industry", "sector", "revenue"]].isnull().sum()

## Step 4 - Drop rows with no job description
We need the description text to find skills in Step 4 of the project, so rows without it are not useful.

In [ ]:
before = len(df)
df = df[df["job_description"].notna() & (df["job_description"].str.len() > 30)].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no usable description -> {len(df)} rows left")

## Step 5 - Group job titles into roles
There are hundreds of different titles. We map each one to a small set of roles. The order of the rules matters: the first rule that matches wins.
Jobs like *Data Entry Operator* and *Data Annotator* are not analyst jobs, so we remove them.
Roles like *Clinical Data Coordinator* or *Data Operations Associate* are kept but labelled **Data Operations**, and a column `is_core_role` marks the roles that matter for this project (analyst, BI, data science, data engineering, ML).

In [ ]:
def role_group(title):
    t = str(title).lower()
    if re.search(r"data entry|annotat|labeling|labelling|typist|back office", t):
        return "Excluded"
    if re.search(r"\bbi\b|business intelligence|power ?bi", t):
        return "BI Analyst"
    if "business analyst" in t:
        return "Business Analyst"
    if "data engin" in t:                       # also catches the typo "Data Enginner"
        return "Data Engineer"
    if "analyst" not in t and re.search(r"\betl\b|data model|data architect|data warehous|big data", t):
        return "Data Engineer"
    if re.search(r"data scien", t):
        return "Data Scientist"
    if re.search(r"machine learning|\bml\b|\bai\b|deep learning|\bnlp\b", t):
        return "ML/AI Engineer"
    if re.search(r"analyst|analytics|analysis", t):
        return "Data Analyst"
    if re.search(r"clinical|operations|coordinator|executive|data management|governance|privacy|security|data associate|research associate|data specialist|\bprocess\b", t):
        return "Data Operations"
    return "Other"

df["role_group"] = df["job_title"].map(role_group)

print(df["role_group"].value_counts())
print()
print("Titles that landed in 'Other' (check these):")
print(df.loc[df["role_group"] == "Other", "job_title"].value_counts().head(15))

In [ ]:
before = len(df)
df = df[df["role_group"] != "Excluded"].reset_index(drop=True)
print(f"Removed {before - len(df)} non-analyst rows -> {len(df)} rows left")

CORE_ROLES = ["Data Analyst", "BI Analyst", "Business Analyst", "Data Scientist", "Data Engineer", "ML/AI Engineer"]
df["is_core_role"] = df["role_group"].isin(CORE_ROLES)
print()
print("Core-role rows:", int(df["is_core_role"].sum()), "of", len(df))
print()
print("Titles still in Other / Data Operations (skim these):")
print(df.loc[~df["is_core_role"], ["job_title", "role_group"]].value_counts().head(25))

## Step 6 - Standardise locations
Problems in the raw data: `Hyderābād` has an accent, `Gurgaon`/`Noida`/`Delhi` are all the Delhi region, `Karnataka` is a state (not a city),
and `Remote` / `India` do not tell us a city. We fix all of these.

In [ ]:
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

CITY_MAP = {
    "bengaluru": "Bengaluru", "bangalore": "Bengaluru",
    "hyderabad": "Hyderabad", "secunderabad": "Hyderabad",
    "pune": "Pune",
    "mumbai": "Mumbai", "thane": "Mumbai", "navi mumbai": "Mumbai",
    "chennai": "Chennai",
    "delhi": "Delhi NCR", "new delhi": "Delhi NCR", "gurgaon": "Delhi NCR", "gurugram": "Delhi NCR",
    "noida": "Delhi NCR", "greater noida": "Delhi NCR", "ghaziabad": "Delhi NCR", "faridabad": "Delhi NCR",
    "kolkata": "Kolkata", "ahmedabad": "Ahmedabad", "jaipur": "Jaipur",
    "cochin": "Kochi", "kochi": "Kochi", "coimbatore": "Coimbatore",
}

NOT_A_CITY = {
    "india", "remote", "karnataka", "maharashtra", "telangana", "tamil nadu", "gujarat", "kerala",
    "haryana", "uttar pradesh", "west bengal", "rajasthan", "andhra pradesh", "madhya pradesh",
    "punjab", "odisha", "bihar", "goa",
}

def clean_city(loc):
    if not isinstance(loc, str):
        return "Not specified"
    first_part = loc.split(",")[0]
    key = strip_accents(first_part).strip().lower()
    if key in NOT_A_CITY or key == "":
        return "Not specified"
    return CITY_MAP.get(key, strip_accents(first_part).strip().title())

df["city"] = df["location"].map(clean_city)
df["work_mode"] = np.where(df["location"].str.strip().str.lower() == "remote", "Remote", "Office/Other")

print(df["city"].value_counts().head(15))

## Step 7 - Convert salary text into numbers
Salary looks like `₹3,25,236` (text, with Indian comma style) and comes with a period: per year, per month or per hour.
We remove the symbol and commas, then convert everything to **annual INR**. Monthly x 12. Hourly rows are dropped (there are only a couple).

Important: these are **Glassdoor estimates**, not confirmed offers. We will say so in the README.

In [ ]:
def parse_salary(x):
    if not isinstance(x, str):
        return np.nan
    if re.search(r"[-\u2013]|\bto\b", x.lower()):   # a range like 3L-5L: skip, we only trust single values
        return np.nan
    digits = re.sub(r"[^\d.]", "", x)
    return float(digits) if digits else np.nan

def period_multiplier(p):
    if not isinstance(p, str):
        return np.nan
    p = p.lower()
    if "/yr" in p:
        return 1
    if "/mo" in p:
        return 12
    return np.nan   # hourly or unknown

df["salary_number"] = df["salary_avg_estimate"].map(parse_salary)
df["salary_annual_inr"] = df["salary_number"] * df["salary_estimate_payperiod"].map(period_multiplier)
df["salary_lpa"] = (df["salary_annual_inr"] / 100000).round(2)   # LPA = lakhs per annum

print("Rows with a usable salary:", df["salary_lpa"].notna().sum(), "of", len(df))
print()
print(df["salary_lpa"].describe())

**Sanity check:** the minimum and maximum above should look believable for Indian analyst/data roles (roughly 1 to 60 LPA).
If you see something like 0.01 or 5000, tell me and we will investigate.

## Step 8 - Work out experience level
There is no experience column, so we get it from two places:
1. The **title**: words like *Intern*, *Junior* mean Entry, and *Senior*, *Lead*, *Manager* mean Senior.
2. The **description**: text like `0 - 2 years of experience`. We take the minimum years mentioned near the word "experience".

This is an estimate, not perfect, and we will say so in the case study.

In [ ]:
EXP_RE = re.compile(r"(?<!\d)(\d{1,2})\s*(?:-|\u2013|to)?\s*(?:\d{1,2})?\s*\+?\s*(?:years?|yrs?)")

def min_experience(text):
    if not isinstance(text, str):
        return np.nan
    t = text.lower()
    for m in re.finditer(r"experience", t):
        window = t[max(0, m.start() - 80): m.end() + 80]
        for found in EXP_RE.finditer(window):
            years = int(found.group(1))
            if years <= 15:   # bigger numbers are usually company history, not a job requirement
                return years
    return np.nan

def level_from_title(title):
    t = str(title).lower()
    if re.search(r"intern|trainee|fresher|junior|\bjr\b|graduate|entry", t):
        return "Entry/Fresher"
    if re.search(r"senior|\bsr\b|lead|principal|manager|head|director|architect|staff|\biii\b|\biv\b", t):
        return "Senior (5+ yrs)"
    return None

def final_level(row):
    if pd.notna(row["level_from_title"]):
        return row["level_from_title"]
    y = row["exp_min_years"]
    if pd.isna(y):
        return "Not specified"
    if y <= 1:
        return "Entry/Fresher"
    if y <= 4:
        return "Mid (2-4 yrs)"
    return "Senior (5+ yrs)"

df["exp_min_years"] = df["job_description"].map(min_experience)
df["level_from_title"] = df["job_title"].map(level_from_title)
df["experience_level"] = df.apply(final_level, axis=1)

# A Junior/Intern role asking for more than 3 years is almost always a false match, so blank it
df.loc[(df["level_from_title"] == "Entry/Fresher") & (df["exp_min_years"] > 3), "exp_min_years"] = np.nan

print(df["experience_level"].value_counts())
print()
print("Minimum years found in descriptions:")
print(df["exp_min_years"].describe())

## Step 9 - Flag suspicious salaries
Glassdoor estimates can be odd. We do **not** delete these rows, we just flag them so averages are not distorted:
- **Too low:** under 1.5 LPA for a job that is not an internship or traineeship (interns are allowed to be low).
- **Too high:** over 30 LPA when nothing suggests a senior role.

`salary_lpa_clean` is the salary with flagged values blanked. We use it for salary charts and keep `salary_lpa` as the original.

In [ ]:
df["is_intern"] = df["job_title"].str.contains(r"intern|trainee", case=False, regex=True)

too_low = (df["salary_lpa"] < 1.5) & ~df["is_intern"]
too_high = (df["salary_lpa"] > 30) & (df["experience_level"] != "Senior (5+ yrs)")

df["salary_suspect"] = too_low | too_high
df["salary_lpa_clean"] = df["salary_lpa"].where(~df["salary_suspect"])

print("Flagged as suspect:", int(df["salary_suspect"].sum()))
print(df.loc[df["salary_suspect"], ["job_title", "experience_level", "salary_lpa"]].sort_values("salary_lpa"))
print()
print("Median salary (LPA) by role, core roles only, flagged rows removed:")
print(df[df["is_core_role"]].groupby("role_group")["salary_lpa_clean"].agg(["count", "median"]).round(2))

## Step 10 - Save the clean file
We keep the useful columns and write the result to `data/clean/jobs_clean.csv`.

In [ ]:
keep = [
    "company", "job_title", "role_group", "is_core_role", "is_intern", "city", "work_mode", "experience_level", "exp_min_years",
    "salary_annual_inr", "salary_lpa", "salary_lpa_clean", "salary_suspect", "company_rating", "company_size", "company_founded",
    "employment_type", "industry", "sector", "revenue", "job_description",
]
clean = df[keep].copy()

clean.to_csv("../data/clean/jobs_clean.csv", index=False, encoding="utf-8")

print("Saved ../data/clean/jobs_clean.csv")
print("Final shape:", clean.shape)
print()
print("Roles:")
print(clean["role_group"].value_counts())
print()
print("Missing values per column:")
print(clean.isnull().sum())

## Checkpoint
Copy the output of the last cell (final shape, roles, missing values) plus the salary and experience summaries and send them back.